In [0]:
from pyspark.sql.functions import count,sum,round,avg
from pyspark.sql import functions as F
silver_df=spark.read.table("dev_nyc.silver.taxi_duartion")

In [0]:
display(silver_df.limit(5))

In [0]:
gold_base_df = silver_df.withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))

# Summary Asset A: Daily Trips & Financial Performance Metrics
daily_trips_gold = (gold_base_df
    .groupBy("pickup_date")
    .agg(
        F.count("VendorID").alias("total_trips"),
        F.sum("passenger_count").alias("total_passengers_carried"),
        F.round(F.sum("trip_distance"), 2).alias("total_distance_miles"),
        F.round(F.sum("fare_amount"), 2).alias("total_fare_revenue_usd"),
        F.round(F.sum("tip_amount"), 2).alias("total_tips_collected_usd"),
        F.round(F.sum("total_amount"), 2).alias("total_gross_revenue_usd"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_minutes"),
        F.round(F.avg("tip_amount") / F.avg("fare_amount") * 100, 2).alias("avg_tip_percentage")
    )
    .orderBy(F.col("pickup_date").desc())
)

In [0]:
vendor_performance_gold = (gold_base_df
    .groupBy("VendorID")
    .agg(
        F.count("VendorID").alias("total_trips_managed"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare_per_trip"),
        F.round(F.max("trip_distance"), 2).alias("longest_trip_mileage"),
        # Breakdown of payment choice selections
        F.count(F.when(F.col("payment_type") == 1, 1)).alias("credit_card_trips"),
        F.count(F.when(F.col("payment_type") == 2, 1)).alias("cash_trips")
    )
)

In [0]:
daily_trips_gold.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("dev_nyc.gold.fact_daily_trips")

vendor_performance_gold.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("dev_nyc.gold.dim_vendor_performance")

print("Gold Layer Business Aggregate tables successfully deployed!")

In [0]:
display(daily_trips_gold.limit(10))

In [0]:
display(vendor_performance_gold.limit(10))

In [0]:
# =====================================================================
# ASSET C: Hourly Traffic & Surge Demand Analysis (Capacity Planning)
# =====================================================================
hourly_demand_gold = (silver_df
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .groupBy("pickup_date", "pickup_hour")
    .agg(
        F.count("VendorID").alias("trip_demand_volume"),
        F.round(F.avg("fare_amount"), 2).alias("avg_base_fare"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
        F.round(F.sum("trip_distance") / (F.sum("trip_duration_minutes") / 60), 2).alias("avg_speed_mph")
    )
    .orderBy("pickup_date", "pickup_hour")
)

hourly_demand_gold.write.format("delta").mode("append").saveAsTable("dev_nyc.gold.fact_hourly_demand")

In [0]:
# =====================================================================
# ASSET E: Vendor Performance Fact Table (Granular Analytical Layer)
# =====================================================================
import pyspark.sql.functions as F

fact_vendor_performance = (silver_df
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .groupBy(
        "pickup_date",
        "pickup_hour",
        "VendorID",
        "PULocationID",
        "DOLocationID"
    )
    .agg(
        F.count("VendorID").alias("trip_count"),
        F.sum("passenger_count").alias("total_passengers"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("fare_amount"), 2).alias("total_base_fare"),
        F.round(F.sum("tip_amount"), 2).alias("total_tips"),
        F.round(F.sum("tolls_amount"), 2).alias("total_tolls"),
        F.round(F.sum("trip_distance"), 2).alias("total_distance_miles"),
        F.round(F.sum("trip_duration_minutes"), 2).alias("total_duration_minutes"),
        F.round(
            F.avg(
                F.when(F.col("trip_duration_minutes") > 0,
                       F.col("trip_distance") / (F.col("trip_duration_minutes") / 60))
                .otherwise(None)
            ), 2
        ).alias("avg_trip_speed_mph"),
        F.round(
            F.avg(
                F.when(F.col("fare_amount") > 0,
                       (F.col("tip_amount") / F.col("fare_amount")) * 100)
                .otherwise(0.0)
            ), 2
        ).alias("avg_tip_percentage")
    )
    .orderBy("pickup_date", "pickup_hour", "VendorID")
)

(fact_vendor_performance.write
    .format("delta")
    .mode("append")
    .partitionBy("pickup_date")
    .saveAsTable("dev_nyc.gold.fact_vendor_performance"))

In [0]:
display(fact_vendor_performance)